In [ ]:
%pip install numpy pandas statsmodels scipy openpyxl lxml matplotlib linearmodels fredapi

In [9]:
import glob
import pandas as pd

total_id = "PAYEMS"
sector_codes = [
    "USMINE", "USCONS", "MANEMP", "USINFO", "USTPU", 
    "USFIRE", "USPBS", "USEHS", "USLAH", "USSERV", "USGOVT"
]

def load_file_series(code):
    matches = glob.glob(f"{code}.*")
    if not matches:
        raise FileNotFoundError(f"File for '{code}' not found.")
    
    filepath = matches[0]
    if filepath.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(filepath)
    else:
        df = pd.read_csv(filepath)
        
    date_col = df.columns[0]
    val_col = df.columns[1]
    
    df["Date"] = pd.to_datetime(df[date_col], errors="coerce")
    df[code] = pd.to_numeric(df[val_col], errors="coerce")
    df = df.dropna(subset=["Date"]).set_index("Date").sort_index()
    return df[[code]]

# Load PAYEMS and all 11 sectors
all_series = [total_id] + sector_codes
df_monthly = pd.concat([load_file_series(code) for code in all_series], axis=1).dropna()
df_monthly = df_monthly.loc["1947-01-01":"2010-01-01"]

# Aggregate monthly observations to quarterly averages
try:
    df_quarterly = df_monthly.resample("QE").mean()
except ValueError:
    df_quarterly = df_monthly.resample("Q").mean()

# Compute employment shares relative to PAYEMS
shares = pd.DataFrame(index=df_quarterly.index)
for code in sector_codes:
    shares[f"{code}_share"] = df_quarterly[code] / df_quarterly[total_id]

# Calculate absolute first differences and index
abs_share_diffs = shares.diff().abs()
sectoral_shifts_index = abs_share_diffs.sum(axis=1).to_frame(name="Sectoral_Shifts_Index")

# Merge and slice sample from 1948Q1
output_df = pd.concat([
    df_quarterly.add_suffix("_level"),
    shares,
    sectoral_shifts_index
], axis=1).dropna().loc["1948-01-01":]

# Standardize date output for load_fred_series compatibility
output_df.index = output_df.index.strftime("%Y-%m-%d")
output_df.to_csv("sectoral_shifts_index.csv", index_label="Quarter")

print(f"Generated sectoral_shifts_index.csv with {len(output_df)} observations.")

Generated sectoral_shifts_index.csv with 249 observations.


#### Consolidation

In [10]:
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# 1. Date Sanitization Routine (Supports CSV and Excel .xlsx transparently)
# -----------------------------------------------------------------------------
def load_fred_series(filepath, val_col_name=None):
    """
    Loads FRED files (.csv or .xlsx).
    Handles Excel native dates or parses two-digit string dates (%m-%d-%y / %d-%b-%y / %Y-%m-%d).
    Corrects two-digit rollover where years >= 2030 are adjusted to 1930-1999.
    Returns a DataFrame with a standardized datetime 'Date' column and 'Quarter'.
    """
    if str(filepath).lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(filepath)
    else:
        df = pd.read_csv(filepath)
        
    date_col = df.columns[0]
    
    # If already parsed as datetime by pandas/openpyxl
    if pd.api.types.is_datetime64_any_dtype(df[date_col]):
        parsed = df[date_col]
    else:
        # Standardize string parsers
        parsed = pd.to_datetime(df[date_col], format="%Y-%m-%d", errors="coerce")
        if parsed.isna().all():
            parsed = pd.to_datetime(df[date_col], format="%m-%d-%y", errors="coerce")
        if parsed.isna().all():
            parsed = pd.to_datetime(df[date_col], format="%d-%b-%y", errors="coerce")
        if parsed.isna().all():
            parsed = pd.to_datetime(df[date_col], errors="coerce")
        
    # Adjust two-digit century rollover (mapping 2030-2099 back to 1930-1999)
    df["Date"] = parsed.apply(
        lambda dt: dt.replace(year=dt.year - 100) if pd.notnull(dt) and dt.year >= 2030 else dt
    )
    df = df.dropna(subset=["Date"]).sort_values("Date")
    df["Quarter"] = df["Date"].dt.to_period("Q")
    
    # Extract value column
    if val_col_name is not None and val_col_name in df.columns:
        df["value"] = pd.to_numeric(df[val_col_name], errors="coerce")
    else:
        non_date_cols = [c for c in df.columns if c not in [date_col, "Date", "Quarter"]]
        df["value"] = pd.to_numeric(df[non_date_cols[0]], errors="coerce")
        
    return df

# -----------------------------------------------------------------------------
# 2. Stage 1 SVAR Series (1948:Q1 - 2001:Q1 to compute differences from 1948:Q2)
# -----------------------------------------------------------------------------
# Real GDP: Quarterly (GDPC1)
df_rgdp = load_fred_series("rgdp_q_us.csv", "GDPC1")
rgdp_q = df_rgdp.groupby("Quarter")["value"].last()

# CPI: Monthly (CPIAUCSL) aggregated to End-of-Quarter (last month)
df_cpi = load_fred_series("cpi_m_us.csv", "CPIAUCSL")
cpi_q = df_cpi.groupby("Quarter")["value"].last()

# Civilian Unemployment Rate: Monthly (UNRATE) aggregated to Quarterly Average
df_ur = load_fred_series("unrate_m_us.csv", "UNRATE")
unrate_q = df_ur.groupby("Quarter")["value"].mean()

# Construct Stage 1 core variables
stage1_df = pd.DataFrame({
    "rgdp": rgdp_q,
    "cpi": cpi_q,
    "unrate": unrate_q
})

# Percent log differences (scaled by 100) and first difference for unemployment
delta_y = 100.0 * np.log(stage1_df["rgdp"] / stage1_df["rgdp"].shift(1))
delta_p = 100.0 * np.log(stage1_df["cpi"] / stage1_df["cpi"].shift(1))
delta_u = stage1_df["unrate"] - stage1_df["unrate"].shift(1)

# -----------------------------------------------------------------------------
# 3. Stage 2 Determinants: Search, Institutional, and Demographic Variables
# -----------------------------------------------------------------------------
# Sectoral Shifts Index
df_shifts = load_fred_series("sectoral_shifts_index.csv", "Sectoral_Shifts_Index")
sectoral_shifts = df_shifts.groupby("Quarter")["value"].last()

# Unemployment Insurance Benefits & Number of Unemployed (Exact FRED Series)
df_ui = load_fred_series("Unemployment_Insurance.xlsx", "W825RC1Q027SBEA")
ui_q = df_ui.groupby("Quarter")["value"].last()  # Billions SAAR

df_unemp = load_fred_series("UNEMPLOY.csv", "UNEMPLOY")
unemp_q = df_unemp.groupby("Quarter")["value"].mean()  # Thousands

# CPI base year 1996 average normalization
cpi_1996_base = cpi_q[cpi_q.index.year == 1996].mean()
real_ui_per_unemp = ((ui_q * 1e9 / 4.0) / (unemp_q * 1e3)) / (cpi_q / cpi_1996_base)
log_real_ui = np.log(real_ui_per_unemp)

# Labor Productivity Growth (Output per hour nonfarm business level index differenced)
df_prod = load_fred_series("Labour_Productivity(Q).csv", "OPHNFB")
prod_q = df_prod.groupby("Quarter")["value"].last()
prod_growth = 100.0 * np.log(prod_q / prod_q.shift(1))

# Real Hourly Compensation (COMPRNFB index level)
df_comp = load_fred_series("real_hourly_comp_q.csv", "COMPRNFB")
comp_q = df_comp.groupby("Quarter")["value"].last()
log_real_comp = np.log(comp_q)

# Pre-1953 Treasury Yields Splicing (NBER Macrohistory m13033a / LTGOVTBD)
pre_1953_yields = {
    "1947Q1": 2.25, "1947Q2": 2.27, "1947Q3": 2.29, "1947Q4": 2.39,
    "1948Q1": 2.42, "1948Q2": 2.43, "1948Q3": 2.44, "1948Q4": 2.44,
    "1949Q1": 2.38, "1949Q2": 2.38, "1949Q3": 2.24, "1949Q4": 2.21,
    "1950Q1": 2.20, "1950Q2": 2.23, "1950Q3": 2.35, "1950Q4": 2.39,
    "1951Q1": 2.40, "1951Q2": 2.63, "1951Q3": 2.57, "1951Q4": 2.62,
    "1952Q1": 2.70, "1952Q2": 2.61, "1952Q3": 2.68, "1952Q4": 2.75,
    "1953Q1": 2.80, "1953Q2": 2.97, "1953Q3": 2.98, "1953Q4": 2.83,
}
historical_series = pd.Series(
    list(pre_1953_yields.values()),
    index=pd.PeriodIndex(list(pre_1953_yields.keys()), freq="Q"),
    name="GS10"
)

# 10-Year Treasury Yield (GS10) spliced with pre-1953 benchmark
df_gs10 = load_fred_series("treasury_10y_yield_m.csv", "GS10")
gs10_q = df_gs10.groupby("Quarter")["value"].mean()
nominal_10y_yield = gs10_q.combine_first(historical_series)

# Ex-post Real Interest Rate (10Y yield minus annual CPI inflation)
annual_cpi_inflation = 100.0 * np.log(cpi_q / cpi_q.shift(4))
real_interest_rate = nominal_10y_yield - annual_cpi_inflation

# Help-Wanted Vacancy Index (barnichon composite, starts 1951Q1)
df_hw = load_fred_series("help_wanted_index_m.csv")
hw_q = df_hw.groupby("Quarter")["value"].mean()
log_help_wanted = np.log(hw_q)

# Real Federal Minimum Wage
df_mw = load_fred_series("min_wage_real_q.csv")
mw_q = df_mw.groupby("Quarter")["value"].last()
log_real_min_wage = np.log(mw_q)

# Union Density: Troy-Sheflin (1948-1963) spliced with CPS 'All States' (1964-2001)
troy_sheflin = pd.Series({
    1948: 30.5, 1949: 29.6, 1950: 30.0, 1951: 32.4, 1952: 31.5,
    1953: 33.2, 1954: 32.7, 1955: 33.2, 1956: 33.4, 1957: 32.8,
    1958: 32.5, 1959: 31.4, 1960: 30.9, 1961: 30.2, 1962: 29.8, 1963: 29.6
})

cps_union = pd.read_excel("union_density_cps_historical.xlsx")
all_states_row = cps_union[cps_union.iloc[:, 0].astype(str).str.contains("All States", case=False, na=False)].iloc[0, 1:]
all_states_row.index = pd.to_numeric(all_states_row.index, errors="coerce")
cps_annual = all_states_row.dropna()

union_annual = pd.concat([troy_sheflin, cps_annual[cps_annual.index >= 1964]]).sort_index()
quarterly_grid = pd.period_range("1948Q1", "2001Q4", freq="Q")
union_df = pd.DataFrame(index=quarterly_grid)
union_df["Year"] = union_df.index.year
union_df["annual_val"] = union_df["Year"].map(union_annual)
union_density = union_df["annual_val"].interpolate(method="linear")

# Civilian Labor Force Growth
df_clf = load_fred_series("CLF16OV.csv", "CLF16OV")
clf_q = df_clf.groupby("Quarter")["value"].mean()
clf_growth = 100.0 * np.log(clf_q / clf_q.shift(1))

# Demographic Employment Ratios
df_ce = load_fred_series("CE16OV.csv", "CE16OV")
ce_q = df_ce.groupby("Quarter")["value"].mean()

df_m20 = load_fred_series("LNS12000001.csv")
df_m16 = load_fred_series("LNS12000014.csv")
df_y16 = load_fred_series("LNS12000012.csv")
df_y20 = load_fred_series("LNS12000036.csv")
df_s55 = load_fred_series("LNS12024230.csv")

m20_q = df_m20.groupby("Quarter")["value"].mean()
m16_q = df_m16.groupby("Quarter")["value"].mean()
male_share = (m20_q + m16_q) / ce_q

y16_q = df_y16.groupby("Quarter")["value"].mean()
y20_q = df_y20.groupby("Quarter")["value"].mean()
youth_share = (y16_q + y20_q) / ce_q

senior_share = df_s55.groupby("Quarter")["value"].mean() / ce_q

# -----------------------------------------------------------------------------
# 4. Master Dataset Construction (1948:Q2 - 2001:Q1, exactly 212 observations)
# -----------------------------------------------------------------------------
master_df = pd.DataFrame({
    "delta_y": delta_y,
    "delta_p": delta_p,
    "delta_u": delta_u,
    "unrate": stage1_df["unrate"],
    "sectoral_shifts": sectoral_shifts,
    "log_real_ui": log_real_ui,
    "prod_growth": prod_growth,
    "log_real_comp": log_real_comp,
    "real_interest_rate": real_interest_rate,
    "log_help_wanted": log_help_wanted,
    "log_real_min_wage": log_real_min_wage,
    "union_density": union_density,
    "clf_growth": clf_growth,
    "male_share": male_share,
    "youth_share": youth_share,
    "senior_share": senior_share
}).loc["1948Q2":"2001Q1"]

# Verify shape and completeness
print("==================================================================")
print(f"Master Dataset Shape: {master_df.shape}")
print(f"Sample Horizon: {master_df.index.min()} to {master_df.index.max()}")
print("Missing values count per variable:")
print(master_df.isna().sum())
print("==================================================================")

# Save master dataset for VAR estimation and Stage 2 regressions
master_df.to_csv("km2007_master_data.csv", index=True)
print("Saved clean master dataset to km2007_master_data.csv")

C:\Users\potat\AppData\Local\Temp\ipykernel_42652\2617071972.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[date_col], errors="coerce")
C:\Users\potat\AppData\Local\Temp\ipykernel_42652\2617071972.py:153: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  union_annual = pd.concat([troy_sheflin, cps_annual[cps_annual.index >= 1964]]).sort_index()


Master Dataset Shape: (212, 16)
Sample Horizon: 1948Q2 to 2001Q1
Missing values count per variable:
delta_y                0
delta_p                0
delta_u                0
unrate                 0
sectoral_shifts        0
log_real_ui            0
prod_growth            0
log_real_comp          0
real_interest_rate     0
log_help_wanted       11
log_real_min_wage      0
union_density          0
clf_growth             0
male_share             0
youth_share            0
senior_share           0
dtype: int64
Saved clean master dataset to km2007_master_data.csv
